# OpenTelemetry Operator

Telemetrie ist der Prozess der automatischen Erfassung und Übertragung von Daten von entfernten oder verteilten Systemen, um deren Leistung und Status zu überwachen, zu messen und zu verfolgen. Telemetriedaten liefern Echtzeit-Einblicke in die Funktionsweise verschiedener Anwendungskomponenten. Sie stellen die Daten für Observability-Tools bereit, die Entwicklern und Systemadministratoren helfen, das System zu beobachten, Fehler zu beheben und es zu optimieren, ohne jede Komponente manuell überprüfen zu müssen.

**OpenTelemetry (OTel) ist ein Open-Source-Projekt**, das standardisierte Tools und APIs zum Generieren, Sammeln und Exportieren von Telemetriedaten wie Traces, Metriken und Logs bereitstellt. Ziel ist es, Entwicklern umfassende Einblicke in Anwendungen zu ermöglichen und sie bei der Überwachung, Fehlerbehebung und Optimierung von Softwaresystemen zu unterstützen.

Die Hauptziele von OpenTelemtry sind:

* Einheitliche Telemetrie : Kombiniert Tracing, Logging und Metriken in einem einzigen Framework, das die Korrelation aller Daten ermöglicht und einen offenen Standard für Telemetriedaten etabliert.
* Anbieterneutralität : Integration mit verschiedenen Backends zur Datenverarbeitung.
* Plattformübergreifend : Unterstützt verschiedene Sprachen (Java, Python, Go usw.) und Plattformen und ist daher vielseitig für unterschiedliche Entwicklungsumgebungen einsetzbar.


---

### Installation

zuerst muss der Cert-Manager installiert werden

In [ ]:
%%bash
curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/main/services/cert-manager.sh | bash -

Als nächstes der OpenTelemetry Operator.

Dieser ermöglicht es Telemetry Daten zu erheben ohne den Applikationscode ändern zu müssen.

In [ ]:
%%bash
helm repo add open-telemetry https://open-telemetry.github.io/opentelemetry-helm-charts
helm upgrade --install opentelemetry-operator open-telemetry/opentelemetry-operator \
  -n opentelemetry \
  --create-namespace

Kontrolle ob alles läuft und CRD `instrumentation` vorhanden ist.

In [ ]:
%%bash
kubectl get pods -n opentelemetry
kubectl get crd | grep instrumentation

Der OpenTelemetryCollector sammelt die Daten und leitet sie an 

    exporters:
      zipkin:
        endpoint: http://zipkin.istio-system.svc.cluster.local:9411/api/v2/spans

weiter        

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1beta1
kind: OpenTelemetryCollector
metadata:
  name: otel-collector
  namespace: opentelemetry
spec:
  mode: daemonset
  image: otel/opentelemetry-collector-contrib:latest
  config:
    receivers:
      otlp:
        protocols:
          grpc:
            endpoint: 0.0.0.0:4317
          http:
            endpoint: 0.0.0.0:4318

    processors:
      memory_limiter:
        check_interval: 1s
        limit_percentage: 75
        spike_limit_percentage: 15
      batch: {}

    exporters:
      zipkin:
        endpoint: http://zipkin.istio-system.svc.cluster.local:9411/api/v2/spans
      debug: {}

    service:
      pipelines:
        traces:
          receivers: [otlp]
          processors: [memory_limiter, batch]
          exporters: [zipkin, debug]
EOF

Kontrolle ob ein OpenTelemetryCollector Pod erstellt wurde

In [ ]:
%%bash
kubectl get opentelemetrycollector -n opentelemetry
kubectl get pods -n opentelemetry
kubectl get svc -n opentelemetry